# 7 ways to automate SERP analysis with AI

Companion notebook for the MLforSEO guide by Lazarina Stoy. It runs end to end on the real sample data in `sample_data/` (no keys needed), then shows where to plug in your own keys.

1. SERP API + scheduled collection
2. LLM structured-output classification of results, intent and SERP features
3. Entity and n-gram gap analysis
4. AI Overview and AI Mode citation tracking
5. Share of search over time
6. Clustering keywords by SERP overlap
7. Scripted and agentic pipelines

All the functions live in `serp_automation.py`, so you can reuse them outside the notebook.

In [ ]:
# In Colab: !pip install -q pandas requests scikit-learn scipy matplotlib spacy anthropic && python -m spacy download en_core_web_sm
# then upload serp_automation.py and the sample_data/ folder (or clone the repo).
import os, pandas as pd, matplotlib.pyplot as plt
import serp_automation as sa
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
DATA_DIR = "sample_data"   # swap for "data" once you collect your own

## 1. SERP API + scheduled collection

`DataForSEO.organic()` pulls the full Google SERP (every feature type, plus the AI Overview), `ai_mode()` pulls Google AI Mode, and `historical()` backfills up to a year of monthly snapshots. `save_snapshot()` appends to CSVs, de-duplicated, so the same call can run every day from cron or GitHub Actions. There is a spend cap on the client so a loop can't run away with your credit.

The cell below only runs when your DataForSEO keys are set. The sample data was collected exactly this way on 22 October 2026 (US, English, desktop).

In [ ]:
if os.environ.get("DATAFORSEO_LOGIN"):
    client = sa.DataForSEO(max_spend_usd=0.05)
    sa.collect(["serp analysis"], client, "data", ai_mode=True)
    print(f"spent ${client.spent:.4f}")
snap = sa.load_snapshots(DATA_DIR)
{k: v.shape for k, v in snap.items()}

In [ ]:
snap["organic"].groupby("date")["domain"].apply(list)

## 2. LLM structured-output classification

The SERP API already types every element (AI Overview, People Also Ask, video...), so feature detection needs no AI at all. Where an LLM earns its place is the judgement layer: what *kind* of page each result is, its intent, and what the searcher mostly wants to do on this SERP. The JSON schema puts the label lists in enums, so the model can only answer from them.

In [ ]:
sa.feature_matrix(snap["features"])

In [ ]:
print(sa.serp_prompt("serp analysis",
      snap["organic"].query("date == '2026-10-22'"), snap["features"].query("date == '2026-10-22'"))[:900])

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    labels = sa.classify_latest(snap)
    display(labels)
else:
    print("Set ANTHROPIC_API_KEY to label the latest SERPs. Schema:", list(sa.CLASSIFY_SCHEMA["properties"]))

## 3. Entity and n-gram gap analysis

Compare what the ranking results (and the pages the AI answers cite) talk about with what your page covers. Here the "SERP corpus" is the titles and snippets of the latest top 10 plus the titles of the cited pages, and "my page" is the MLforSEO SERP guide. For a deeper version, scrape the full pages first.

In [ ]:
latest = snap["organic"].query("date == date.max()")
cited = snap["citations"].query("date == date.max()")["title"].drop_duplicates()
serp_docs = (latest["title"] + ". " + latest["description"].fillna("")).tolist() + cited.tolist()
my_page = open(f"{DATA_DIR}/my_page.txt").read()
ngrams = sa.ngram_gap(serp_docs, my_page, min_share=0.2)
ngrams

In [ ]:
ents = sa.entity_gap(serp_docs, my_page, min_docs=2)   # spaCy; pass extractor=sa.entities_google for Google NL
ents[~ents["in_my_page"]].head(15)

## 4. AI Overview and AI Mode citation tracking

In [ ]:
sa.citation_overlap(snap["citations"], snap["organic"])

In [ ]:
pres = sa.citation_presence(snap["citations"], "serp analysis")
pres[pres.sum(axis=1) >= 2]

In [ ]:
sa.my_citation_rate(snap["citations"], "mlforseo.com")

## 5. Share of search over time

Slot share counts positions; click-weighted share multiplies each position by `sa.CTR_CURVE`, which is an assumption. Replace it with your own Search Console CTR curve.

In [ ]:
sos = sa.share_of_search(snap["organic"], weight="ctr")
sos.T[sos.T.max(axis=1) >= 0.05].round(2)

In [ ]:
sa.biggest_movers(sos)

In [ ]:
top = sos.T[sos.T.max(axis=1) >= 0.15].T
ax = (top * 100).plot(marker="o", figsize=(9, 4.5), title='Click-weighted share of the top 10: "serp analysis"')
ax.set_ylabel("%"); ax.legend(bbox_to_anchor=(1, 1)); plt.tight_layout()

## 6. Clustering keywords by SERP overlap

If two keywords share several top-10 URLs, Google treats them as the same need, and one page can usually target both. Here: three SERP keywords from 22 October 2026, then the six machine-learning keywords from the older sample file.

In [ ]:
ov_kw = pd.read_csv(f"{DATA_DIR}/overlap_keywords.csv")
both = pd.concat([latest[ov_kw.columns], ov_kw])
overlap = sa.serp_overlap(both)
display(overlap)
sa.cluster_by_overlap(overlap, min_shared=3)

In [ ]:
ml = pd.read_csv("sample_serp_data.csv").rename(columns={"position": "rank_group"}).assign(date="2026-10-22")
sa.serp_overlap(ml)

## 7. Scripted and agentic pipelines

* `run_pipeline.py` is the scripted version: collect, store, measure, write `report.md`. Schedule it with `github-actions-serp-tracker.yml`.
* `serp_agent.py` gives Claude read-only tools over the same snapshots (plus one capped, paid live-pull tool) for the open question on top: what changed, and what should we do?

In [ ]:
print(sa.change_report(snap, my_domain="mlforseo.com"))

In [ ]:
# !python run_pipeline.py --demo --data-dir demo_out
# !python serp_agent.py --data-dir sample_data --my-domain mlforseo.com   # needs ANTHROPIC_API_KEY